In [0]:
#Setup

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}"

)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"

)

print("Schemas Gold and Silver available")


In [0]:
#Functions

def read_bronze(table_name: str) -> DataFrame:
    """It reads a bronze table"""
    return spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

def save_silver(
    dataframe: DataFrame,
    table_name: str,
) -> None: 
    """It saves a dataframe as a delta table on Silver"""

    target_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    )

    (   
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(target_table)

    )
    row_count = dataframe.count()

    print(
        f"Table saved: {target_table} "
        f"({row_count:,} rows)"
    )


In [0]:
#CUSTOMERS

#Read the customers_bronze tables
customers_bronze = read_bronze("olist_customers")
display(customers_bronze.limit(5))
customers_bronze.printSchema()

#Clean the data

window_spec = Window.partitionBy("customer_id").orderBy(F.desc("_ingested_at"))

customers_silver = (
    customers_bronze
    .select(
        F.trim("customer_id").alias("customer_id"),
        F.trim("customer_unique_id").alias("customer_unique_id"),
        F.col("customer_zip_code_prefix").cast("int").alias("customer_zip_code_prefix"),
        F.initcap(F.trim("customer_city")).alias("customer_city"),
        F.upper(F.trim("customer_state")).alias("customer_state"),
        F.col("_ingested_at")
    )
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("customer_id").isNotNull())
    .dropna(how='any')
)

#Validation

display(customers_silver.limit(10))

print(
    "Bronze rows:",
        customers_bronze.count(),
)
print(
    "Silver rows:",
        customers_silver.count()

)

print(
    "Distinct customer_id's:",
    customers_silver
    .select("customer_id")
    .distinct()
    .count()

)

#Save Customers Silver Table

save_silver(
    customers_silver,
    "customers",

)


In [0]:
#ORDERS

#Read the orders_bronze tables
orders_bronze = read_bronze("olist_orders")
display(orders_bronze.limit(5))
orders_bronze.printSchema()

#Clean the data

window_spec = Window.partitionBy("order_id").orderBy(F.desc("_ingested_at"))

orders_silver = (
    orders_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.trim("customer_id").alias("customer_id"),
        F.lower(F.trim("order_status")).alias("order_status"),
        F.to_timestamp(F.trim("order_purchase_timestamp")).alias("purchased_at"),
        F.to_timestamp(F.trim("order_approved_at")).alias("approved_at"),
        F.to_timestamp(F.trim("order_delivered_carrier_date")).alias("delivered_to_carrier_at"),
        F.to_timestamp(F.trim("order_delivered_customer_date")).alias("delivered_to_customer_at"),
        F.to_timestamp(F.trim("order_estimated_delivery_date")).alias("estimated_delivery_at"),
        F.col("_ingested_at"),
    )
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("order_id").isNotNull())
    .dropna(how='any')
)

#Validation

display(orders_silver.limit(10))

print(
    "Bronze rows:",
        orders_bronze.count(),
)
print(
    "Silver rows:",
        orders_silver.count()

)

print(
    "Distinct order_id's:",
    orders_silver
    .select("order_id")
    .distinct()
    .count()

)

#Save ORDERS Silver Table

save_silver(
    orders_silver,
    "orders",

)


In [0]:
#ORDER_ITEMS

#Read the order_items bronze tables
order_items_bronze = read_bronze("olist_order_items")
display(order_items_bronze.limit(5))
order_items_bronze.printSchema()

#Clean the data

order_items_silver = (
    order_items_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.col("order_item_id").cast("integer").alias("order_item_sequence"),
        F.trim("product_id").alias("product_id"),
        F.trim("seller_id").alias("seller_id"),
        F.to_timestamp("shipping_limit_date").alias("shipping_limit_at"),
        F.round(F.col("price").cast("double"), 2,).alias("item_price"),
        F.round(F.col("freight_value").cast("double"), 2,).alias("freight_value"),
        F.col("_ingested_at"),
    )
    .withColumn("_row_num", F.row_number().over(
        Window.partitionBy("order_id", "order_item_sequence").orderBy(F.desc("_ingested_at"))
    ))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("item_price").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .dropna(how='any')
)

#Validation

display(order_items_silver.limit(10))

print(
    "Bronze rows:",
        order_items_bronze.count(),
)
print(
    "Silver rows:",
        order_items_silver.count()

)

print(
    "Distinct order_id's:",
    order_items_silver
    .select("order_id")
    .distinct()
    .count()

)

#Save ORDERS Silver Table

save_silver(
    order_items_silver,
    "order_items",

)


In [0]:
#PAYMENTS

payments_bronze = read_bronze("olist_order_payments")
display(payments_bronze.limit(500))

payments_silver = (payments_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.col("payment_sequential").cast("integer").alias("payment_sequential"),
        F.lower(F.trim("payment_type")).alias("payment_type"),
        F.col("payment_installments").cast("integer").alias("payment_installments"),
        F.round(F.col("payment_value").cast("double"), 2).alias("payment_value"),
    )
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("payment_value") >= 0)
    .dropDuplicates(["order_id", "payment_value", "payment_sequential"])
    .dropna(how='any')

)

#AGGREGATION: PAYMENTS BY ORDER

payments_by_order = (payments_silver
    .groupBy("order_id")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
        F.max("payment_installments").alias("max_installments"),
        F.sort_array(F.collect_set("payment_type")).alias("payment_methods"),
        F.count("*").alias("payment_record_count"),

    )

)

save_silver(payments_by_order, "payments_by_order")




In [0]:
%sql
SELECT order_id, payment_type, payment_value
FROM workspace.ecommerce_silver.payments
LIMIT 20;

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_bronze.olist_products
WHERE product_id LIKE "46b48281eb6d663ced748f324108c733"
LIMIT 20

In [0]:
#PRODUCTS

products_bronze = read_bronze("olist_products")
translation_bronze = read_bronze("product_category_translation")

category_translation = (translation_bronze
    .select(
        F.lower(F.trim("product_category_name")).alias("category_name_pt"),
        F.lower(F.trim("product_category_name_english")).alias("category_name_eng"),
    )
    .dropDuplicates(["category_name_pt"])
)                    

products_silver = (products_bronze
    .select(
        F.trim("product_id").alias("product_id"),
        F.lower(F.trim("product_category_name")).alias("category_name_pt"),
        F.col("product_name_lenght").cast("integer").alias("product_name_length"),
        F.col("product_description_lenght").cast("integer").alias("product_description_length"),
        F.col("product_photos_qty").cast("integer").alias("product_photos_qty"),
        F.col("product_weight_g").cast("integer").alias("product_weight_g"),
        F.col("product_length_cm").cast("integer").alias("product_length_cm"),
        F.col("product_height_cm").cast("integer").alias("product_height_cm"),
        F.col("product_width_cm").cast("integer").alias("product_width_cm"),

    )
    .dropDuplicates(["product_id"])
    .filter(F.col("product_id").isNotNull())
                        
                        
)

products_silver = (products_silver
    .join(category_translation,
        on="category_name_pt",
        how="left",
    )
    .withColumn("category_name_eng", F.coalesce(F.col("category_name_eng"), F.lit("unknown"))
    )
    .dropna(subset=['product_id', 'product_weight_g'])     
          
)

save_silver(products_silver, "products")         
  

In [0]:
#SELLERS

sellers_bronze = read_bronze("olist_sellers")
display(sellers_bronze.limit(5))

window_spec = Window.partitionBy("seller_id").orderBy(F.desc("_ingested_at"))

sellers_silver = (
    sellers_bronze
    .select(
        F.trim("seller_id").alias("seller_id"),
        F.col("seller_zip_code_prefix").cast("integer").alias("seller_zip_code_prefix"),
        F.initcap(F.trim("seller_city")).alias("seller_city"),
        F.upper(F.trim("seller_state")).alias("seller_state"),
        F.col("_ingested_at")
    )
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("seller_id").isNotNull())
    .dropna(how='any')
)

print(
    "Bronze rows:",
    sellers_bronze.count(),
)
print(
    "Silver rows:",
    sellers_silver.count()
)
print(
    "Distinct seller_id's:",
    sellers_silver.select("seller_id").distinct().count()
)

display(sellers_silver.limit(10))
save_silver(sellers_silver, "sellers")

In [0]:
#ORDER REVIEWS

reviews_bronze = read_bronze("olist_order_reviews")
display(reviews_bronze.limit(5))

window_spec = Window.partitionBy("review_id").orderBy(F.desc("_ingested_at"))

reviews_silver = (
    reviews_bronze
    # Filter only valid scores (1-5) to avoid malformed data
    .filter(F.col("review_score").isin("1", "2", "3", "4", "5"))
    .select(
        F.trim("review_id").alias("review_id"),
        F.trim("order_id").alias("order_id"),
        F.col("review_score").cast("integer").alias("review_score"),
        F.trim("review_comment_title").alias("review_comment_title"),
        F.trim("review_comment_message").alias("review_comment_message"),
        # Use try_cast to handle malformed timestamps (returns NULL)
        F.expr("try_cast(review_creation_date as timestamp)").alias("review_created_at"),
        F.expr("try_cast(review_answer_timestamp as timestamp)").alias("review_answered_at"),
        F.col("_ingested_at")
    )
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
    .filter(F.col("review_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("review_score").isNotNull())
    .dropna(subset=["review_id", "order_id", "review_score", "review_created_at"])
)

# Validation
print(
    "Bronze rows:",
    reviews_bronze.count(),
)
print(
    "Silver rows:",
    reviews_silver.count()
)
print(
    "Distinct review_id's:",
    reviews_silver.select("review_id").distinct().count()
)
print(
    "Reviews with comments:",
    reviews_silver.filter(F.col("review_comment_message").isNotNull()).count()
)

display(reviews_silver.limit(10))
save_silver(reviews_silver, "order_reviews")

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.products
LIMIT 20    

In [0]:
#PRODUCTS FOR RAG

products_for_rag = (
    spark.table("workspace.ecommerce_silver.products")
    .filter(F.col("category_name_pt").isNotNull())
    .withColumn(
        "product_name_length",
        F.coalesce(F.col("product_name_length"), F.lit(0))
    )
    .withColumn(
        "product_description_length",
        F.coalesce(F.col("product_description_length"), F.lit(0))
    )
    .withColumn(
        "product_photos_qty",
        F.coalesce(F.col("product_photos_qty"), F.lit(0))
    )
    
    #Quality score for RAG
    .withColumn(
        "data_completeness_score",
        (
            F.when(F.col("category_name_pt").isNotNull(), 1).otherwise(0) +
            F.when(F.col("product_name_length") > 0, 1).otherwise(0) +
            F.when(F.col("product_description_length") > 0, 1).otherwise(0) +
            F.when(F.col("product_photos_qty") > 0, 1).otherwise(0)
        ).cast("integer")
    )
    
    #Searchable text field for embeddings (in Portuguese for RAG)
    .withColumn(
        "searchable_text",
        F.concat_ws(
            " | ",
            F.coalesce(F.col("category_name_pt"), F.lit("desconhecido")),
            F.concat(F.lit("peso: "), F.col("product_weight_g"), F.lit("g")),
            F.concat(F.lit("dimensões: "), F.col("product_length_cm"), F.lit("x"), 
                     F.col("product_width_cm"), F.lit("x"), F.col("product_height_cm"), F.lit("cm"))
        )
    )
)

print(f"\nRAG Quality Analysis:")
print(f"Silver products (all): {spark.table('workspace.ecommerce_silver.products').count():,}")
print(f"RAG-ready products: {products_for_rag.count():,}")
print(f"Filtered out: {spark.table('workspace.ecommerce_silver.products').count() - products_for_rag.count():,}")

print("\nCompleteness Distribution:")
display(products_for_rag.groupBy("data_completeness_score").count().orderBy("data_completeness_score"))

# Preview searchable text
print("\nSearchable Text Examples:")
display(products_for_rag.select("product_id", "category_name_pt", "searchable_text").limit(10))